# MLflow Experiment Tracking & Model Reproducibility

## Objective

In Notebook 05, I refitted the provisional logistic T-learner on all 200,039 labeled customers and used it to score the 200,123 customers in the unlabeled X5 population.

The model and decision engine are working, but the saved model artifact alone does not provide a complete record of its provenance, feature schema, or relationship to earlier experiments.

In this notebook, I introduce MLflow to record the full-data model, its configuration, its input features, and the data and code versions associated with it.

I also verify that a model loaded from MLflow reproduces the predictions from the original saved artifact.

**Important distinction:** The full-data model is used for scoring. The development-validation metrics from the earlier model-evaluation notebook belong to models fitted on a smaller training subset. I will document those metrics separately rather than incorrectly attributing them to the full-data refit.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from mlflow.models import infer_signature



In [ ]:
# ============================================================
# Setup and local MLflow configuration
#
# MLflow uses two different storage locations:
#
# Tracking database:
#   Stores experiment names, run IDs, parameters, metrics,
#   artifact references, and other metadata.
#
# Artifact directory:
#   Stores the actual serialized model files and documents.
#
# Both locations are local development resources and are
# excluded from Git.
# ============================================================

# ------------------------------------------------------------
# Locate the repository root without relying on the
# notebook's current working directory.
# ------------------------------------------------------------

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dbt" / "dbt_project.yml").exists()
)


# ------------------------------------------------------------
# Define the existing inputs from Notebook 05.
# ------------------------------------------------------------

MODEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "models"
    / "logistic_t_learner_full.joblib"
)

TRAINING_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_uplift_training.parquet"
)

SCORING_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_uplift_scoring.parquet"
)

PREDICTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "uplift_scoring_predictions.parquet"
)


# ------------------------------------------------------------
# Put the SQLite database and artifacts in one clearly
# defined location.
#
# The database URI uses four slashes for an absolute
# Unix/macOS file path.
# ------------------------------------------------------------

MLFLOW_DIR = PROJECT_ROOT / "data" / "mlflow"

ARTIFACT_DIR = MLFLOW_DIR / "artifacts"

TRACKING_DB = MLFLOW_DIR / "mlflow.db"

MLFLOW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRACKING_URI = (
    f"sqlite:///{TRACKING_DB.as_posix()}"
)

mlflow.set_tracking_uri(TRACKING_URI)


# ------------------------------------------------------------
# Create the experiment only if it does not already exist.
#
# Repeated notebook runs should not create a new experiment
# with a slightly different name each time.
# ------------------------------------------------------------

EXPERIMENT_NAME = "retail_growth_uplift"

experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

if experiment is None:

    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=ARTIFACT_DIR.as_uri(),
    )

mlflow.set_experiment(
    EXPERIMENT_NAME
)


print(f"MLflow version: {mlflow.__version__}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Tracking database: {TRACKING_DB}")
print(f"Artifact directory: {ARTIFACT_DIR}")

## 2. Load the existing fitted model

Notebook 05 already produced a full-data logistic T-learner. I will reuse that saved artifact rather than fitting another model simply to create an MLflow record.

The T-learner consists of two independent pipelines:

- A treatment-outcome model fitted on treated customers.
- A control-outcome model fitted on control customers.

Each pipeline includes its own preprocessing and logistic-regression estimator.

I will verify that the saved feature list matches the current scoring dataset and record file fingerprints so that the tracking record identifies the exact local inputs used.

A file fingerprint does not establish that the data is correct. It identifies a particular file version and helps detect unexpected changes.

In [ ]:

# ============================================================
# Load the trusted model artifact and verify its inputs
#
# joblib uses Python object serialization.
# Only load artifacts created by you or another trusted
# source; never load an untrusted model file.
# ============================================================

REQUIRED_FILES = [
    MODEL_PATH,
    TRAINING_PATH,
    SCORING_PATH,
    PREDICTION_PATH,
]

missing_files = [
    path
    for path in REQUIRED_FILES
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Required files are missing: {missing_files}"
    )


# ------------------------------------------------------------
# Load the exact fitted models and feature list produced
# by Notebook 05.
# ------------------------------------------------------------

model_bundle = joblib.load(
    MODEL_PATH
)

treatment_model = model_bundle[
    "treatment_model"
]

control_model = model_bundle[
    "control_model"
]

feature_columns = model_bundle[
    "feature_columns"
]


# ------------------------------------------------------------
# Load the scoring data and the predictions previously
# generated from these fitted models.
# ------------------------------------------------------------

scoring_df = pd.read_parquet(
    SCORING_PATH
)

scores = pd.read_parquet(
    PREDICTION_PATH
)

scoring_df.columns = scoring_df.columns.str.lower()
scores.columns = scores.columns.str.lower()


# ------------------------------------------------------------
# Validate the model's feature contract.
#
# Column order matters because the fitted preprocessing
# pipelines were trained using a particular feature order.
# ------------------------------------------------------------

assert len(feature_columns) == len(set(feature_columns))

missing_features = (
    set(feature_columns)
    - set(scoring_df.columns)
)

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

assert len(scoring_df) == 200_123
assert len(scores) == 200_123

assert scoring_df["client_id"].is_unique
assert scores["client_id"].is_unique

assert scoring_df["client_id"].reset_index(
    drop=True
).equals(
    scores["client_id"].reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Calculate SHA-256 file fingerprints.
#
# Read files in chunks instead of loading the entire file
# into memory simply to calculate a hash.
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


fingerprints = {
    "training_parquet_sha256":
        sha256_file(TRAINING_PATH),

    "scoring_parquet_sha256":
        sha256_file(SCORING_PATH),

    "predictions_parquet_sha256":
        sha256_file(PREDICTION_PATH),

    "original_model_sha256":
        sha256_file(MODEL_PATH),
}


# ------------------------------------------------------------
# Record the current Git commit, if one is available.
#
# The commit identifies source code, not the ignored local
# Parquet files or model artifacts.
# ------------------------------------------------------------

git_commit = subprocess.run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


print(f"Scoring customers: {len(scoring_df):,}")
print(f"Model features: {len(feature_columns)}")
print(f"Git commit: {git_commit}")
print("Model and data inputs verified.")

## 3. Log the full-data model to MLflow

I will create one MLflow run representing the full-data logistic T-learner used for scoring.

The run records:

- Model family and training population
- Feature cutoff and feature-column order
- Source-code commit and local data fingerprints
- Treatment and control model artifacts
- Input schema and a representative input example

The two outcome models are logged separately because the T-learner requires both predictions to calculate uplift.

I do not attach the earlier development-validation ROC-AUC, Qini, or uplift@30% measurements as performance metrics for this full-data refit. Those measurements were obtained from different fitted model instances and belong to a separate evaluation record.

In [ ]:

# ============================================================
# Create the full-data scoring-model MLflow run
#
# This records existing fitted models.
#
# No retraining occurs in this cell.
# ============================================================

X_scoring = scoring_df[
    feature_columns
]


# ------------------------------------------------------------
# Use a small representative input example to document
# the model's expected input structure.
#
# The example contains customer FEATURES ONLY.
# It does not include client_id, treatment, or target.
# ------------------------------------------------------------

X_example = X_scoring.head(100).copy()


# ------------------------------------------------------------
# Infer input/output signatures using the probability
# interface that our decision engine actually needs.
# ------------------------------------------------------------

treatment_signature = infer_signature(
    X_example,
    treatment_model.predict_proba(
        X_example
    ),
)

control_signature = infer_signature(
    X_example,
    control_model.predict_proba(
        X_example
    ),
)


# ------------------------------------------------------------
# Build a JSON-compatible feature contract.
#
# It preserves feature order and records the source dataset
# columns used by the fitted models.
# ------------------------------------------------------------

feature_contract = {
    "model_family": "logistic_t_learner",
    "feature_columns": list(feature_columns),
    "feature_dtypes": {
        column: str(X_scoring[column].dtype)
        for column in feature_columns
    },
    "feature_count": len(feature_columns),
    "feature_cutoff": model_bundle["feature_cutoff"],
    "training_population": 200_039,
    "scoring_population": 200_123,
    "prediction_definition":
        "P(target=1 | X,T=1) - P(target=1 | X,T=0)",
}


# ------------------------------------------------------------
# Start the run and store the model's provenance.
#
# We intentionally do not log credentials, .env contents,
# customer-level datasets, or raw customer identifiers.
# ------------------------------------------------------------

with mlflow.start_run(
    run_name="full_data_logistic_t_learner"
) as run:

    mlflow.set_tags(
        {
            "purpose": "unlabeled_population_scoring",
            "evaluation_status":
                "no_independent_full_refit_test",
            "source_notebook":
                "05_treatment_decisioning.ipynb",
            "git_commit": git_commit,
            "causal_status":
                "conditional_on_unverified_assignment_assumptions",
        }
    )

    mlflow.log_params(
        {
            "model_family": "logistic_t_learner",
            "estimator": "LogisticRegression",
            "max_iter": 2000,
            "training_customers": 200_039,
            "scoring_customers": 200_123,
            "feature_count": len(feature_columns),
            "feature_cutoff":
                model_bundle["feature_cutoff"],
        }
    )

    mlflow.log_dict(
        feature_contract,
        "metadata/feature_contract.json",
    )

    mlflow.log_dict(
        fingerprints,
        "metadata/input_fingerprints.json",
    )


    # --------------------------------------------------------
    # Log treatment and control as separate native sklearn
    # models with explicit probability-prediction interfaces.
    #
    # cloudpickle is appropriate for our trusted local
    # sklearn pipelines. Never deserialize untrusted models.
    # --------------------------------------------------------

    treatment_info = mlflow.sklearn.log_model(
        sk_model=treatment_model,
        name="treatment_outcome_model",
        signature=treatment_signature,
        input_example=X_example,
        pyfunc_predict_fn="predict_proba",
        serialization_format="cloudpickle",
    )

    control_info = mlflow.sklearn.log_model(
        sk_model=control_model,
        name="control_outcome_model",
        signature=control_signature,
        input_example=X_example,
        pyfunc_predict_fn="predict_proba",
        serialization_format="cloudpickle",
    )


    # --------------------------------------------------------
    # Preserve both model references in one small manifest.
    # These references will be useful for the API later.
    # --------------------------------------------------------

    model_manifest = {
        "mlflow_run_id": run.info.run_id,
        "treatment_model_uri":
            treatment_info.model_uri,
        "control_model_uri":
            control_info.model_uri,
        "feature_count": len(feature_columns),
        "git_commit": git_commit,
    }

    mlflow.log_dict(
        model_manifest,
        "metadata/model_manifest.json",
    )


print(f"MLflow run ID: {model_manifest['mlflow_run_id']}")
print(f"Treatment model: {model_manifest['treatment_model_uri']}")
print(f"Control model: {model_manifest['control_model_uri']}")

## 4. Verify model reload and prediction consistency

A reproducibility record is only useful if its artifacts can actually be loaded and used.

I reload both outcome models from their MLflow model URIs and compare their probability predictions against the original fitted models on the same customer sample.

I also verify that the combined uplift predictions match the predictions saved in Notebook 05.

This is a functional artifact test. It checks serialization and prediction consistency in the current environment, but does not by itself establish that the model will reproduce identically under every future dependency version or operating system.

In [ ]:

# ============================================================
# MLflow artifact verification
#
# Load the actual logged artifacts rather than relying only
# on the original joblib models still held in memory.
# ============================================================

reloaded_treatment_model = (
    mlflow.sklearn.load_model(
        model_manifest["treatment_model_uri"]
    )
)

reloaded_control_model = (
    mlflow.sklearn.load_model(
        model_manifest["control_model_uri"]
    )
)


# ------------------------------------------------------------
# Compare predictions on the same 100 scoring customers.
# ------------------------------------------------------------

original_treatment_proba = (
    treatment_model.predict_proba(
        X_example
    )[:, 1]
)

reloaded_treatment_proba = (
    reloaded_treatment_model.predict_proba(
        X_example
    )[:, 1]
)

original_control_proba = (
    control_model.predict_proba(
        X_example
    )[:, 1]
)

reloaded_control_proba = (
    reloaded_control_model.predict_proba(
        X_example
    )[:, 1]
)


np.testing.assert_allclose(
    original_treatment_proba,
    reloaded_treatment_proba,
    rtol=1e-12,
    atol=1e-12,
)

np.testing.assert_allclose(
    original_control_proba,
    reloaded_control_proba,
    rtol=1e-12,
    atol=1e-12,
)


# ------------------------------------------------------------
# Also compare reloaded uplift predictions against the
# prediction cache created in Notebook 05.
# ------------------------------------------------------------

reloaded_uplift = (
    reloaded_treatment_proba
    - reloaded_control_proba
)

np.testing.assert_allclose(
    reloaded_uplift,
    scores["predicted_uplift"].iloc[:100].to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)


# ------------------------------------------------------------
# Store a small local manifest so later notebooks and the
# API can locate the same exact MLflow model artifacts.
#
# This file is ignored by Git because it points to a local
# development tracking database.
# ------------------------------------------------------------

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "models"
    / "mlflow_model_manifest.json"
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH.write_text(
    json.dumps(
        model_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


print("PASS: Both MLflow models reloaded successfully.")
print("PASS: Treatment probabilities match.")
print("PASS: Control probabilities match.")
print("PASS: Uplift predictions match Notebook 05.")
print(f"Model manifest saved: {MANIFEST_PATH}")

## 5. Record the development-validation benchmark

The model-evaluation notebook compared logistic and boosted T-learners on the same 50,010-customer development holdout.

Those results informed the provisional decision to use the logistic T-learner for the targeting simulator.

I preserve the reported results in a separate MLflow reference run. This run is a historical evaluation summary, not a new training run and not an independent assessment of the full-data scoring models.

The metrics below are transcribed from the completed model-evaluation notebook. The original notebook remains the source for the training procedure, predictions, plots, and calculations.

In [ ]:

# ============================================================
# Historical development-validation reference
#
# These metrics describe the earlier development models.
#
# They must NOT be interpreted as newly measured metrics
# for the full-data model logged above.
# ============================================================

development_benchmark = {
    "evaluation_population": 50_010,
    "evaluation_type": "development_validation",
    "independent_final_test": False,

    "logistic_t_learner": {
        "treatment_auc": 0.765062,
        "control_auc": 0.772814,
        "treatment_brier": 0.186098,
        "control_brier": 0.188313,
        "qini_coefficient": 168.366182,
        "auuc": 0.061673,
        "uplift_at_30pct": 0.063582,
        "uplift_30_ci_low": 0.048655,
        "uplift_30_ci_high": 0.078112,
    },

    "boosted_t_learner": {
        "treatment_auc": 0.776690,
        "control_auc": 0.782102,
        "treatment_brier": 0.181828,
        "control_brier": 0.184455,
        "qini_coefficient": 137.729506,
        "auuc": 0.062355,
        "uplift_at_30pct": 0.059267,
        "uplift_30_ci_low": 0.041237,
        "uplift_30_ci_high": 0.076408,
    },

    "interpretation": (
        "Development-validation comparison only. "
        "Causal interpretation requires additional "
        "treatment-assignment assumptions."
    ),
}


with mlflow.start_run(
    run_name="development_validation_reference"
) as reference_run:

    mlflow.set_tags(
        {
            "run_type": "historical_metrics_reference",
            "source": "uplift_model_evaluation_notebook",
            "metrics_origin":
                "transcribed_from_previous_notebook_outputs",
            "independent_final_test": "false",
        }
    )

    mlflow.log_param(
        "evaluation_customers",
        development_benchmark["evaluation_population"],
    )

    mlflow.log_dict(
        development_benchmark,
        "metadata/development_benchmark.json",
    )

    for model_name in [
        "logistic_t_learner",
        "boosted_t_learner",
    ]:

        model_metrics = development_benchmark[
            model_name
        ]

        mlflow.log_metrics(
            {
                f"{model_name}_{metric_name}": metric_value
                for metric_name, metric_value
                in model_metrics.items()
            }
        )


print(
    "Development benchmark recorded separately."
)

print(
    f"Reference run ID: {reference_run.info.run_id}"
)

## 6. Conclusions and limitations

I introduced MLflow to track the full-data logistic T-learner used in the treatment decisioning platform.

Rather than retraining the model unnecessarily, I loaded the existing fitted artifact from Notebook 05 and recorded its configuration, feature schema, training population, feature cutoff, Git commit, and local data fingerprints.

I logged the treatment and control outcome models as separate MLflow artifacts and verified that both could be reloaded successfully. The reloaded models reproduced the original treatment, control, and uplift predictions on the verification sample.

I also recorded the earlier logistic-versus-boosted model comparison in a separate development-validation reference run. This preserves the experimental history without incorrectly attributing those performance metrics to the full-data refit.

### Limitations

The MLflow tracking database and model artifacts are stored locally. They are suitable for development but are not yet a shared or production-hosted model registry.

The prediction-consistency test verifies the artifacts in the current environment. Reproducibility across other environments will also require controlled dependencies and deployment configuration.

The full-data refit has not been evaluated on an independent labeled test set. Its use in the decision simulator remains based on the exploratory development-validation results.

The original X5 treatment-assignment mechanism remains unverified, so predicted uplift and modeled economic value should not be presented as independently verified causal effects or realized profits.

The next step is to expose the tracked models and reusable decision engine through an API.